# Neurodesk for Multiple Sclerosis

**Author:** micmas (Neurodesk MS Workshop)

**Date:** June 2026

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

**Note:** This notebook uses neuroimaging tools from Neurocontainers. Those tools keep their original licenses — see the [Neurodesk citation guidelines](https://neurodesk.org/overview/how-to-cite-us/).

---

This notebook runs a small Multiple Sclerosis (MS) imaging pipeline from start to finish inside Neurodesk. It loads the tools, downloads an open MS dataset, and walks through the main analyses one step at a time. Run the cells from top to bottom.

You can run it in [Neurodesk Play](https://play.neurodesk.org/) (in your browser, nothing to install) or in any other Neurodesk environment.

The slow steps are **timed**, and **skipped automatically** if their results already exist — so pointing the working directory at a folder with pre-computed results lets you skip the waiting.

## Citation and Resources

**Tools used**
- FSL (SIENA, FLIRT): Smith et al., NeuroImage 2004; Jenkinson et al., NeuroImage 2012.
- FreeSurfer (SAMSEG): Puonti, Iglesias & Van Leemput, NeuroImage 2016.
- LST-AI: Wiltgen et al., NeuroImage: Clinical 2024.
- Spinal Cord Toolbox (SCT): De Leener et al., NeuroImage 2017.
- Neurodesk: Renton, Dao, Johnstone et al., Nature Methods 2024.

**Dataset**
- open_ms_data (Ljubljana 3D MS lesion benchmark): Lesjak et al., Neuroinformatics 2018. https://github.com/muschellij2/open_ms_data

## Table of contents
[1. Load the tools](#1.-Load-the-tools)  
[2. Get the data](#2.-Get-the-data)  
[3. Look at the data](#3.-Look-at-the-data)  
[4. Lesion segmentation](#4.-Lesion-segmentation-with-SAMSEG)  
[5. Brain atrophy](#5.-Brain-atrophy-with-SIENA)  
[6. Group analysis](#6.-Group-analysis-across-subjects)  
[7. Going further](#7.-Going-further:-spinal-cord-and-quantitative-MRI)

## 1. Load the tools

Neurodesk provides each neuroimaging package as a module. `module.load()` makes a tool available inside this notebook, so the cells below (including `!shell` commands) can use it. Here we load FSL and FreeSurfer.

In [1]:
# Load the neuroimaging tools we need into this notebook.
import module
await module.load('fsl/6.0.7.22')
await module.load('freesurfer/8.2.0')
await module.list()

['fsl/6.0.7.22', 'freesurfer/8.2.0']

In [2]:
# Environment, Python packages, and small helpers used throughout the notebook.
import os, sys, time, subprocess, importlib
from pathlib import Path
from contextlib import contextmanager

os.environ['FSLOUTPUTTYPE'] = 'NIFTI_GZ'

import warnings
warnings.filterwarnings('ignore', message='.*no sform.*')  # open_ms_data masks lack an sform; harmless here

# Install anything the kernel is missing.
for pkg in ['nibabel', 'nilearn', 'pandas', 'ipyniivue', 'pyreadr']:
    if importlib.util.find_spec(pkg) is None:
        print('installing', pkg, '…')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

import nibabel as nib
import numpy as np
import pandas as pd
from nilearn.image import resample_to_img
from ipyniivue import NiiVue
from IPython.display import display
import matplotlib.pyplot as plt
%matplotlib inline

# --- helpers ---------------------------------------------------------------
TIMINGS = {}

@contextmanager
def timed(label):
    """Time a processing step and remember how long it took."""
    print(f'▶ {label} …')
    t0 = time.time()
    yield
    dt = time.time() - t0
    TIMINGS[label] = dt
    print(f'✓ {label}: {dt:.1f} s ({dt/60:.1f} min)')

def have(*paths):
    """True if every expected output already exists (so the step can be skipped)."""
    return all(Path(p).exists() for p in paths)

def viewer(volumes):
    """Interactive NiiVue viewer. `volumes`: list of dicts with 'path' (+ optional 'colormap', 'opacity')."""
    nv = NiiVue(back_color=(1, 1, 1, 1), show_3D_crosshair=True)
    nv.load_volumes([{**v, 'path': str(v['path'])} for v in volumes])
    return nv

print('Setup complete.')

Setup complete.


### Configuration

Set the working directory once, here. Everything (data and results) lives under it.

- **Default** — your own copy of the repo. The data is downloaded and the results are written here the first time you run the analyses.
- **Workshop** — switch `WORK` to the shared teaching folder, which already contains the pre-processed results. The analysis cells detect them and skip the slow processing.

In [3]:
# ============================ Configuration ============================
WORK = Path('/home/jovyan/neurodesktop-storage/neurodesk-ms-workshop')
# WORK = Path('/data/teaching/micmas/neurodesk-ms-workshop')

DATA = WORK / 'open_ms_data'      # input dataset (downloaded in section 2)
OUT  = WORK / 'derivatives'       # analysis results
OUT.mkdir(parents=True, exist_ok=True)

print('WORK :', WORK)
print('DATA :', DATA)
print('OUT  :', OUT)

WORK : /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop
DATA : /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/open_ms_data
OUT  : /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/derivatives


In [4]:
# Confirm the command-line tools actually respond
!flirt -version
!run_samseg --help 2>&1 | head -1
!echo "FREESURFER_HOME=$FREESURFER_HOME"

FLIRT version 6.0
usage: run_samseg [-h] -o DIR -i FILE [FILE ...] [-m MODE [MODE ...]]
FREESURFER_HOME=/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_8.2.0_20260601/freesurfer_8.2.0_20260601.simg/opt/freesurfer-8.2.0


## 2. Get the data

We use an open MS dataset. It has a cross-sectional set (one scan per patient, with an expert lesion mask) and a longitudinal set (two timepoints per patient) — enough for the lesion and atrophy analyses. The download is a few hundred MB and runs once.

In [5]:
# Download the open MS dataset once (a few hundred MB).
if DATA.exists():
    print('Dataset already present:', DATA)
else:
    with timed('download dataset'):
        !cd {WORK} && git clone --depth 1 https://github.com/muschellij2/open_ms_data.git
!ls {DATA}

Dataset already present: /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/open_ms_data
README.Rmd	 cs_coreg.rda  long_coreg.rda  longitudinal	   programs
README.md	 cs_demog.rda  long_demog.rda  makefile
cross_sectional  cs_raw.rda    long_raw.rda    open_ms_data.Rproj


In [6]:
# Locate the files robustly, without hard-coding deep paths.
def pick_dir(*candidates):
    for c in candidates:
        if Path(c).is_dir():
            return Path(c)
    return None

def grab(folder, *keys):
    """First .nii.gz in `folder` whose name contains all `keys` (case-insensitive)."""
    for f in sorted(Path(folder).glob('*.nii.gz')):
        n = f.name.lower()
        if all(k in n for k in keys):
            return f
    return None

def find_gt(folder):
    return (grab(folder, 'consensus') or grab(folder, 'gold') or
            grab(folder, 'lesion') or grab(folder, 'gt') or grab(folder, 'mask'))

cs = DATA / 'cross_sectional'
cs_root = pick_dir(cs/'coregistered_resampled', cs/'coregistered', cs)   # prefer smaller resampled
patients = sorted(p for p in cs_root.iterdir() if p.is_dir())
print(f'{len(patients)} cross-sectional patients under {cs_root.name}/')

# Default subject: pin a name, or auto-pick the first with T1 + FLAIR + expert mask
# so every step (segmentation, Dice, viewers) runs fully.
SUBJECT = 'patient03'   # e.g. 'patient01' to force one; None = auto-select a complete subject
if SUBJECT:
    subj = cs_root / SUBJECT
else:
    subj = next((p for p in patients if grab(p, 't1') and grab(p, 'flair') and find_gt(p)),
                patients[0])

print('Using subject:', subj.name)
print('Files:', [f.name for f in sorted(subj.glob("*.nii.gz"))])

FLAIR = grab(subj, 'flair')
T1    = grab(subj, 't1')
GT    = find_gt(subj)
for label, f in [('FLAIR', FLAIR), ('T1', T1), ('lesion GT', GT)]:
    print(f'{label:10}: {f}')
assert FLAIR and T1 and GT, 'Subject is missing T1/FLAIR/expert mask - set SUBJECT to another patient.'


30 cross-sectional patients under coregistered_resampled/
Using subject: patient03
Files: ['FLAIR.nii.gz', 'T1W.nii.gz', 'T1WKS.nii.gz', 'T2W.nii.gz', 'brainmask.nii.gz', 'consensus_gt.nii.gz']
FLAIR     : /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/open_ms_data/cross_sectional/coregistered_resampled/patient03/FLAIR.nii.gz
T1        : /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/open_ms_data/cross_sectional/coregistered_resampled/patient03/T1W.nii.gz
lesion GT : /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/open_ms_data/cross_sectional/coregistered_resampled/patient03/consensus_gt.nii.gz


## 3. Look at the data

MS lesions are bright on FLAIR and darker on T1. Let's view the T1, the FLAIR, and the expert lesion mask for one patient.

In [7]:
# Interactive viewers (drag = move crosshair, scroll = zoom).
print('T1 — lesions look iso/dark:')
display(viewer([{'path': T1, 'colormap': 'gray'}]))

print('FLAIR — lesions are bright; expert mask overlaid in red:')
vols = [{'path': FLAIR, 'colormap': 'gray'}]
if GT:
    vols.append({'path': GT, 'colormap': 'red', 'opacity': 0.5})
display(viewer(vols))

T1 — lesions look iso/dark:


FLAIR — lesions are bright; expert mask overlaid in red:


## 4. Lesion segmentation with SAMSEG

SAMSEG (FreeSurfer) segments brain structures and white-matter lesions from any mix of contrasts — no training or GPU needed. We give it the T1 and the FLAIR. This takes about 10–20 minutes.

In [8]:
SAMSEG = OUT / subj.name / 'samseg'
if have(SAMSEG / 'seg.mgz'):
    print('SAMSEG results found — skipping (delete the folder to re-run).')
else:
    SAMSEG.mkdir(parents=True, exist_ok=True)
    with timed('SAMSEG'):
        !run_samseg --input {T1} {FLAIR} --lesion --lesion-mask-pattern 0 1 --threads 4 --output {SAMSEG}

SAMSEG results found — skipping (delete the folder to re-run).


In [9]:
# Total lesion volume is reported in samseg.stats
stats_file = SAMSEG / 'samseg.stats'
if stats_file.exists():
    for line in stats_file.read_text().splitlines():
        if 'lesion' in line.lower():
            print(line)
else:
    print('samseg.stats not found — did SAMSEG finish? Check the cell output above.')

# Measure Lesions, 18.372640, mm^3


In [10]:
# Binarize SAMSEG's lesion label (99), then score it against the expert mask.
lesions = SAMSEG / 'lesions.nii.gz'
if not have(lesions):
    with timed('SAMSEG: binarize lesions'):
        !mri_binarize --i {SAMSEG}/seg.mgz --match 99 --o {lesions}

pred = nib.load(str(lesions))
pred_d = pred.get_fdata() > 0
print('SAMSEG lesion voxels:', int(pred_d.sum()))

if GT:
    gt = resample_to_img(str(GT), pred, interpolation='nearest')
    gt_d = gt.get_fdata() > 0
    inter = np.logical_and(pred_d, gt_d).sum()
    denom = pred_d.sum() + gt_d.sum()
    dice = (2.0 * inter / denom) if denom else float('nan')
    print(f'Dice vs expert consensus mask: {dice:.3f}')

SAMSEG lesion voxels: 3
Dice vs expert consensus mask: 0.005


In [11]:
# Interactive view: SAMSEG lesions (red) on FLAIR.
display(viewer([
    {'path': FLAIR, 'colormap': 'gray'},
    {'path': SAMSEG / 'lesions.nii.gz', 'colormap': 'red', 'opacity': 0.6},
]))

### Compare with LST-AI

LST-AI is a deep-learning tool trained on MS data. It segments lesions from T1 + FLAIR and also labels them by brain region. Neurodesk provides it as the `lstai` module. On a CPU it takes roughly 15–40 minutes (it skull-strips with HD-BET and runs three networks) and downloads its model weights on the first run.

In [12]:
# Load LST-AI (deep-learning MS lesion segmentation).
await module.load('lstai/1.2.0')

# Quick check that the command is available.
!lst --help 2>&1 | head -5

###########################

Thank you for using LST-AI. If you publish your results, please cite our paper:
Wiltgen T, McGinnis J, Schlaeger S, Kofler F, Voon C, Berthele A, Bischl D, Grundl L, Will N, Metz M, Schinz D, Sepp D, Prucker P, Schmitz-Koep B, Zimmer C, Menze B, Rueckert D, Hemmer B, Kirschke J, Mühlau M, Wiestler B. LST-AI: A Deep Learning Ensemble for Accurate MS Lesion Segmentation. NeuroImage: Clinical, Volume 42, 2024. https://doi.org/10.1016/j.nicl.2024.103611.
###########################


In [13]:
LSTOUT = OUT / subj.name / 'lst_ai'
LSTTMP = OUT / subj.name / 'lst_ai_tmp'

if any(LSTOUT.glob('*.nii.gz')):
    print('LST-AI results found — skipping (delete the folder to re-run).')
else:
    LSTOUT.mkdir(parents=True, exist_ok=True); LSTTMP.mkdir(parents=True, exist_ok=True)
    # Images are NOT skull-stripped, so LST-AI runs HD-BET first (default).
    # If your inputs are already brain-extracted, add  --stripped .
    with timed('LST-AI'):
        !lst --t1 {T1} --flair {FLAIR} --output {LSTOUT} --temp {LSTTMP} --device cpu

print('\nLST-AI output files:')
for f in sorted(LSTOUT.iterdir()):
    print(' ', f.name)

LST-AI results found — skipping (delete the folder to re-run).

LST-AI output files:
  annotated_lesion_stats.csv
  lesion_stats.csv
  space-flair_desc-annotated_seg-lst.nii.gz
  space-flair_seg-lst.nii.gz


In [14]:
# Load the LST-AI mask, report volume + Dice, and show its per-region annotation.
lst_seg = grab(LSTOUT, 'seg') or grab(LSTOUT, 'lesion') or next(iter(sorted(LSTOUT.glob('*.nii.gz'))), None)
print('LST-AI segmentation:', lst_seg)

lst_img = nib.load(str(lst_seg))
lst_d = lst_img.get_fdata() > 0
print('LST-AI lesion voxels:', int(lst_d.sum()))

if GT:
    gt_l = resample_to_img(str(GT), lst_img, interpolation='nearest').get_fdata() > 0
    inter = np.logical_and(lst_d, gt_l).sum(); denom = lst_d.sum() + gt_l.sum()
    dice_lst = (2.0 * inter / denom) if denom else float('nan')
    print(f'Dice (LST-AI vs expert): {dice_lst:.3f}')

csv = next(iter(sorted(LSTOUT.glob('*.csv'))), None)
if csv:
    print('\nLesion statistics (', csv.name, '):')
    print(csv.read_text())

LST-AI segmentation: /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/derivatives/patient03/lst_ai/space-flair_desc-annotated_seg-lst.nii.gz
LST-AI lesion voxels: 596
Dice (LST-AI vs expert): 0.571

Lesion statistics ( annotated_lesion_stats.csv ):
Region,Num_Lesions,Num_Vox,Lesion_Volume
Periventricular,4,434,434.0
Juxtacortical,4,18,18.0
Subcortical,6,109,109.0
Infratentorial,1,35,35.0



In [15]:
# Side-by-side interactive views: expert / SAMSEG / LST-AI, each over FLAIR.
if GT:
    print('Expert consensus:')
    display(viewer([{'path': FLAIR, 'colormap': 'gray'}, {'path': GT, 'colormap': 'red', 'opacity': 0.6}]))
print('SAMSEG:')
display(viewer([{'path': FLAIR, 'colormap': 'gray'}, {'path': SAMSEG/'lesions.nii.gz', 'colormap': 'red', 'opacity': 0.6}]))
print('LST-AI:')
display(viewer([{'path': FLAIR, 'colormap': 'gray'}, {'path': lst_seg, 'colormap': 'red', 'opacity': 0.6}]))

# Summary table
flair_img = nib.load(str(FLAIR))
def on_flair(p):
    return resample_to_img(str(p), flair_img, interpolation='nearest').get_fdata() > 0
gt_on = on_flair(GT) if GT else None
print(f"\n{'method':10}{'lesion voxels':>15}{'Dice vs expert':>16}")
if gt_on is not None:
    print(f"{'EXPERT':10}{int(gt_on.sum()):>15}{'1.000':>16}")
for name, p in [('SAMSEG', SAMSEG/'lesions.nii.gz'), ('LST-AI', lst_seg)]:
    m = on_flair(p)
    d = 2*np.logical_and(m, gt_on).sum()/(m.sum()+gt_on.sum()) if gt_on is not None else float('nan')
    print(f"{name:10}{int(m.sum()):>15}{round(d,3):>16}")

Expert consensus:


SAMSEG:


LST-AI:



method      lesion voxels  Dice vs expert
EXPERT               1142           1.000
SAMSEG                  3           0.005
LST-AI                596           0.571


### Clinical interpretation
- **Total lesion volume** is a standard MS biomarker, though it correlates only weakly with disability (the *clinico-radiological paradox*).
- **Lesion location** (periventricular, juxtacortical, infratentorial, spinal) carries diagnostic weight under the **McDonald 2017** criteria — note LST-AI annotates these automatically.
- **SAMSEG vs LST-AI:** SAMSEG is a generative atlas model (no MS training, very robust to protocol changes); LST-AI is a discriminative ensemble trained on MS data (often higher Dice in-domain, but more sensitive to acquisition shifts). Comparing them on your own data is the point of this section.
- **Dice ~0.6–0.7** vs a single rater is typical for automated MS tools, and expert *inter-rater* Dice itself ranges only ~0.5–0.9 — there is no single "true" mask to score against [1, 2].

**References**
1. Commowick O, et al. *Objective evaluation of multiple sclerosis lesion segmentation using a data management and processing infrastructure.* Scientific Reports 8, 13650 (2018). https://doi.org/10.1038/s41598-018-31911-7
2. García-Lorenzo D, et al. *Review of automatic segmentation methods of multiple sclerosis white matter lesions on conventional MRI.* Medical Image Analysis 17(1):1–18 (2013). https://doi.org/10.1016/j.media.2012.09.004

## 5. Brain atrophy with SIENA

SIENA (FSL) measures the percentage brain volume change (PBVC) between two timepoints — a common measure in MS. We run it on the two timepoints of one longitudinal subject. (For reference: healthy ageing is about -0.2%/year; MS is often -0.5 to -1.0%/year.)

In [16]:
lg = DATA / 'longitudinal'
lg_root = pick_dir(lg/'coregistered_resampled', lg/'coregistered', lg)
lg_patients = sorted(p for p in lg_root.iterdir() if p.is_dir())
print(f'{len(lg_patients)} longitudinal patients under {lg_root.name}/')

def two_timepoints(p):
    """Return (T1_tp1, T1_tp2) for a longitudinal patient, or (None, None)."""
    studies = sorted(s for s in p.iterdir() if s.is_dir())
    if len(studies) >= 2:
        return grab(studies[0], 't1'), grab(studies[1], 't1')
    t1s = sorted(p.rglob('*[tT]1*.nii.gz'))
    return (t1s[0], t1s[1]) if len(t1s) >= 2 else (None, None)

# Pick the first longitudinal subject that actually has two T1 timepoints, so SIENA runs fully.
lsubj, T1_tp1, T1_tp2 = lg_patients[0], None, None
for p in lg_patients:
    tp1, tp2 = two_timepoints(p)
    if tp1 and tp2:
        lsubj, T1_tp1, T1_tp2 = p, tp1, tp2
        break

print('Using longitudinal subject:', lsubj.name)
print('TP1 T1:', T1_tp1)
print('TP2 T1:', T1_tp2)
assert T1_tp1 and T1_tp2, 'Could not find two T1 timepoints - inspect the longitudinal folder layout.'


20 longitudinal patients under coregistered/
Using longitudinal subject: patient01
TP1 T1: /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/open_ms_data/longitudinal/coregistered/patient01/study1_T1W.nii.gz
TP2 T1: /home/jovyan/neurodesktop-storage/neurodesk-ms-workshop/open_ms_data/longitudinal/coregistered/patient01/study2_T1W.nii.gz


In [17]:
SIENA = OUT / lsubj.name / 'siena'
report = SIENA / 'report.siena'
if have(report):
    print('SIENA results found — skipping (delete the folder to re-run).')
else:
    SIENA.mkdir(parents=True, exist_ok=True)
    with timed('SIENA'):
        !siena {T1_tp1} {T1_tp2} -o {SIENA}

SIENA results found — skipping (delete the folder to re-run).


In [18]:
report = SIENA / 'report.siena'
if report.exists():
    txt = report.read_text()
    print(txt)
    for line in txt.splitlines():
        if 'PBVC' in line:
            print('>>>', line.strip())
else:
    print('report.siena not found — check the SIENA output above (and report.html for QC).')

-----------------------------------------------------------------------

 SIENA - Structural Image Evaluation, using Normalisation, of Atrophy
 part of FSL www.fmrib.ox.ac.uk/fsl
 running longitudinal atrophy measurement: siena version 2.6
 siena 

----------  extract brain  --------------------------------------------

----------  register brains and skulls  -------------------------------
(do not worry about histogram warnings)

----------  produce valid masks  --------------------------------------

----------  change analysis  ------------------------------------------
corr1=.9229125846 corr2=1.1051997030 corr=1.0140561438

/opt/fsl-6.0.7.22/bin/flirt -o B_halfwayto_A -applyisoxfm 1 -paddingsize 0 -init B_halfwayto_A.mat -ref B -in B
/opt/fsl-6.0.7.22/bin/flirt -o A_halfwayto_B -applyisoxfm 1 -paddingsize 0 -init A_halfwayto_B.mat -ref B -in A
/opt/fsl-6.0.7.22/bin/flirt -o B_halfwayto_A_mask -applyisoxfm 1 -paddingsize 0 -init B_halfwayto_A.mat -ref B -in B_brain_mask
/opt/fsl-6.0

### Clinical interpretation
- **Annualised PBVC** is the headline atrophy metric in MS DMT trials.
- **Pseudoatrophy** early in treatment (resolving inflammatory oedema) can confound short intervals — prefer ≥ 12-month follow-up.
- Single-timepoint alternative: **SIENAX** (normalised brain volume). For regional/cortical detail: the **FreeSurfer longitudinal** stream.

## 6. Group analysis across subjects

So far we analysed one subject. Here we repeat the lesion segmentation for a few subjects, compare the two methods as a group, and then bring in the **clinical data** shipped with the dataset (age, sex, MS type, and the **EDSS** disability score) to relate lesion load to disability.

> Running SAMSEG + LST-AI for several subjects takes hours on CPU. The loop reuses the skip-if-exists logic, so once the results are pre-computed (e.g. in the shared teaching folder) this section just reads them.

In [19]:
# Process the first few subjects, reusing any results that already exist.
N_SUBJECTS = 3

def sh(cmd):
    subprocess.run(cmd, shell=True, check=False)

def dice(a, b):
    inter = np.logical_and(a, b).sum(); denom = a.sum() + b.sum()
    return 2.0 * inter / denom if denom else float('nan')

def volume_ml(mask_bool, img):
    vox = float(np.prod(img.header.get_zooms()[:3]))
    return mask_bool.sum() * vox / 1000.0

def process_subject(pdir):
    name  = pdir.name
    t1    = grab(pdir, 't1'); flair = grab(pdir, 'flair')
    gt    = grab(pdir, 'consensus') or grab(pdir, 'lesion') or grab(pdir, 'mask')

    # SAMSEG (run if missing, else reuse)
    sam = OUT / name / 'samseg'
    if not have(sam / 'seg.mgz'):
        sam.mkdir(parents=True, exist_ok=True)
        with timed(f'SAMSEG {name}'):
            sh(f'run_samseg --input {t1} {flair} --lesion --lesion-mask-pattern 0 1 --threads 4 --output {sam}')
    sam_les = sam / 'lesions.nii.gz'
    if not have(sam_les):
        sh(f'mri_binarize --i {sam}/seg.mgz --match 99 --o {sam_les}')

    # LST-AI (run if missing, else reuse)
    lst = OUT / name / 'lst_ai'; lsttmp = OUT / name / 'lst_ai_tmp'
    if not any(lst.glob('*.nii.gz')):
        lst.mkdir(parents=True, exist_ok=True); lsttmp.mkdir(parents=True, exist_ok=True)
        with timed(f'LST-AI {name}'):
            sh(f'lst --t1 {t1} --flair {flair} --output {lst} --temp {lsttmp} --device cpu')
    lst_seg = grab(lst, 'seg') or grab(lst, 'lesion') or next(iter(sorted(lst.glob('*.nii.gz'))), None)

    # metrics: Dice in expert space, volume from each native mask
    gt_img = nib.load(str(gt)); gt_d = gt_img.get_fdata() > 0
    row = {'id': name, 'expert_ml': round(volume_ml(gt_d, gt_img), 1)}
    for method, mpath in [('SAMSEG', sam_les), ('LST-AI', lst_seg)]:
        m_img = nib.load(str(mpath)); m_d = m_img.get_fdata() > 0
        m_in_gt = resample_to_img(str(mpath), gt_img, interpolation='nearest').get_fdata() > 0
        row[method + '_dice'] = round(dice(m_in_gt, gt_d), 3)
        row[method + '_ml']   = round(volume_ml(m_d, m_img), 1)
    return row

rows = [process_subject(p) for p in patients[:N_SUBJECTS]]
group = pd.DataFrame(rows)
group

,id,expert_ml,SAMSEG_dice,SAMSEG_ml,LST-AI_dice,LST-AI_ml
0,patient01,31.4,0.498,13.8,0.718,24.1
1,patient02,1.4,0.333,0.3,0.580,0.7
2,patient03,1.1,0.005,0.0,0.571,0.6


In [20]:
# Group comparison: mean Dice per method, plus a boxplot across subjects.
summary = pd.DataFrame({
    'SAMSEG': [group['SAMSEG_dice'].mean(), group['SAMSEG_dice'].std()],
    'LST-AI': [group['LST-AI_dice'].mean(), group['LST-AI_dice'].std()],
}, index=['mean Dice', 'sd']).round(3)
print(summary.to_string())

fig, ax = plt.subplots(figsize=(5, 4))
ax.boxplot([group['SAMSEG_dice'], group['LST-AI_dice']], labels=['SAMSEG', 'LST-AI'])
for i, col in enumerate(['SAMSEG_dice', 'LST-AI_dice'], start=1):
    ax.scatter([i] * len(group), group[col], color='black', alpha=0.6, zorder=3)
ax.set_ylabel('Dice vs expert')
ax.set_title(f'Lesion segmentation accuracy (n={len(group)})')
plt.show()

           SAMSEG  LST-AI
mean Dice   0.279   0.623
sd          0.251   0.082


TypeError: Axes.boxplot() got an unexpected keyword argument 'labels'. Did you mean 'label'?

### Clinical layer: lesion load vs disability (EDSS)

The dataset ships demographic and clinical data per patient (`cs_demog.rda`): age, sex, MS type, and the **EDSS** disability score. We join it to the lesion volumes above and ask whether lesion load tracks disability.

In [ ]:
# Join per-subject lesion volumes to the clinical data shipped with the dataset.
import pyreadr
demog = pyreadr.read_r(str(DATA / 'cs_demog.rda'))['cs_demog']
clin = group.merge(demog, on='id', how='left')
display(clin[['id', 'age', 'sex', 'ms_type', 'edss', 'expert_ml', 'SAMSEG_ml', 'LST-AI_ml']])

sub = clin.dropna(subset=['edss'])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(sub['edss'], sub['LST-AI_ml'])
axes[0].set_xlabel('EDSS (disability)'); axes[0].set_ylabel('LST-AI lesion volume (mL)')
axes[0].set_title('Lesion load vs disability')
if len(sub) >= 3:
    r = sub['edss'].corr(sub['LST-AI_ml'], method='spearman')
    axes[0].text(0.05, 0.95, f'Spearman r = {r:.2f}', transform=axes[0].transAxes, va='top')

order = [t for t in ['CIS', 'RR', 'SP', 'PP'] if t in clin['ms_type'].dropna().unique()]
axes[1].boxplot([clin.loc[clin['ms_type'] == t, 'LST-AI_ml'].dropna() for t in order], labels=order)
axes[1].set_xlabel('MS type'); axes[1].set_ylabel('LST-AI lesion volume (mL)')
axes[1].set_title('Lesion load by MS subtype')
plt.tight_layout(); plt.show()

This is the **clinico-radiological paradox** in miniature: lesion volume relates only loosely to EDSS. With a handful of subjects the correlation is noisy, so treat it as a teaching illustration rather than a result — with the full 30-subject set (pre-processed) the picture is clearer.

## 7. Going further: spinal cord and quantitative MRI

The open dataset has no spinal cord or magnetisation-transfer data, so the two cells below only run if you add your own data at the paths shown. Both are important in MS.

In [ ]:
# Spinal cord cross-sectional area (CSA) with SCT — runs only if you add a cord T2.
CORD_T2  = WORK / 'mydata' / 'cord_T2w.nii.gz'
CORD_OUT = OUT / 'cord'
CSA      = CORD_OUT / 'CSA.csv'
if not CORD_T2.exists():
    print('No cord scan found. Drop a cervical T2 at', CORD_T2, 'to run SCT.')
    print('Upper-cervical (C2–C3) CSA is the most replicated cord-atrophy biomarker in MS.')
elif have(CSA):
    print('Cord CSA results found — skipping.')
    display(pd.read_csv(CSA))
else:
    await module.load('spinalcordtoolbox')   # if this errors: await module.avail('spinal')
    CORD_OUT.mkdir(parents=True, exist_ok=True)
    with timed('SCT cord CSA'):
        # Newer SCT: `sct_deepseg spinalcord`; older SCT: `sct_deepseg_sc`
        !sct_deepseg_sc -i {CORD_T2} -c t2 -ofolder {CORD_OUT}
        SEG = CORD_OUT / (CORD_T2.stem.replace('.nii','') + '_seg.nii.gz')
        !sct_label_vertebrae -i {CORD_T2} -s {SEG} -c t2 -ofolder {CORD_OUT}
        !sct_process_segmentation -i {SEG} -vert 2:5 -perlevel 1 -o {CSA}
    display(pd.read_csv(CSA))

In [ ]:
# Magnetisation Transfer Ratio (MTR), a myelin-sensitive measure — runs only if you add MT data.
QDIR = WORK / 'mydata' / 'qmri'
mt_on, mt_off = QDIR / 'MTon.nii.gz', QDIR / 'MToff.nii.gz'
mtr_out = QDIR / 'MTR.nii.gz'
if not (mt_on.exists() and mt_off.exists()):
    print('No MT data found. Add MTon.nii.gz / MToff.nii.gz under', QDIR, 'to compute MTR.')
elif have(mtr_out):
    print('MTR map found — skipping.')
else:
    with timed('MTR'):
        on  = nib.load(str(mt_on)).get_fdata()
        off_img = nib.load(str(mt_off)); off = off_img.get_fdata()
        with np.errstate(divide='ignore', invalid='ignore'):
            mtr = np.where(off > 0, 100.0 * (off - on) / off, 0.0)
        nib.save(nib.Nifti1Image(mtr, off_img.affine, off_img.header), str(mtr_out))
    print('Saved MTR map. Healthy NAWM MTR ~ 30-40%; lesional MTR drops 20-50% (demyelination).')

## Wrap-up & reproducibility

You ran a full MS imaging mini-pipeline in one environment: **lesion segmentation**
scored against an expert mask, and **longitudinal atrophy** — plus hooks for cord and
qMRI work.

To make a run reproducible:
- **Pin the Neurodesk image tag** (e.g. `vnmd/neurodesktop:2026-04-01`, not `latest`).
- **Pin module versions** — replace `await module.load('fsl')` with the exact version you used (see `await module.list()`).
- Commit this notebook together with those version strings.

**Where to go next:** swap in your own (de-identified) T1+FLAIR, compare SAMSEG against LST or a deep-learning model, or extend the longitudinal analysis to SIENAX / FreeSurfer-longitudinal.

In [ ]:
# How long each step took this run (skipped steps don't appear).
if TIMINGS:
    print('Run times this session:')
    for k, v in TIMINGS.items():
        print(f'  {k:24} {v:8.1f} s  ({v/60:5.1f} min)')
else:
    print('No steps were run this session (all results were already present).')